# 🚗 Parking Intelligence – Data Cleaning Pipeline
**Bengaluru Traffic Violation Records**  
Smart India Hackathon · Data Prep Notebook

---
**Steps covered:**
1. Load & first look
2. Drop useless columns
3. Deduplication
4. Geospatial cleaning
5. Datetime parsing & validation
6. JSON array parsing → multi-label binarization
7. Text / categorical normalization
8. Derived features
9. Null audit & final summary

In [1]:
# ── dependencies ──────────────────────────────────────────────────────────────
import json, re
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1 · Load & First Look

In [2]:
# ── CHANGE THIS PATH to your actual file ──────────────────────────────────────
DATA_PATH = '/kaggle/input/datasets/abhisheknair1984/gridlockv1/jan to may police violation_anonymized791b166.csv'   # CSV, JSON, or Parquet — adjust reader below

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
# df_raw = pd.read_json(DATA_PATH)      # if JSON
# df_raw = pd.read_parquet(DATA_PATH)   # if Parquet

print(f'Shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

Shape  : 298,450 rows × 24 columns


,id,latitude,longitude,location,vehicle_number,vehicle_type,description,violation_type,offence_code,created_datetime,closed_datetime,modified_datetime,device_id,created_by_id,center_code,police_station,data_sent_to_scita,junction_name,action_taken_timestamp,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp
0,FKID000000,12.9256,77.6187,"18th Main Road, Block 2, Koramangala, Bengaluru, Karnata...",FKN00GL0000,CAR,NaN,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,NaN,2023-11-28 04:48:04.582978+00,FKDEV00000,FKUSR00000,9.0000,Madiwala,True,No Junction,NaN,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00
1,FKID000001,12.9055,77.7008,"Sarjapura Main Road, The Grove, Janatha Colony, Doddakan...",FKN00GL0001,CAR,NaN,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00,NaN,2023-11-24 23:00:24.115257+00,FKDEV00001,FKUSR00001,82.0000,Bellandur,False,No Junction,NaN,NaN,NaN,NaN,NaN,NaN
2,FKID000002,12.9254,77.6185,"Koramangala 2nd Block, Kormangala West, Bengaluru South ...",FKN00GL0002,CAR,NaN,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,NaN,2023-11-28 04:47:02.33776+00,FKDEV00000,FKUSR00000,9.0000,Madiwala,True,No Junction,NaN,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00


In [3]:
# column overview: dtype + non-null count + null %
overview = pd.DataFrame({
    'dtype'    : df_raw.dtypes,
    'non_null' : df_raw.notna().sum(),
    'null_pct' : (df_raw.isna().sum() / len(df_raw) * 100).round(2)
})
print(overview.to_string())

                                dtype  non_null  null_pct
id                             object    298450    0.0000
latitude                      float64    298450    0.0000
longitude                     float64    298450    0.0000
location                       object    295409    1.0200
vehicle_number                 object    298450    0.0000
vehicle_type                   object    298450    0.0000
description                   float64         0  100.0000
violation_type                 object    298450    0.0000
offence_code                   object    298450    0.0000
created_datetime               object    298450    0.0000
closed_datetime               float64         0  100.0000
modified_datetime              object    298450    0.0000
device_id                      object    298450    0.0000
created_by_id                  object    298445    0.0000
center_code                   float64    287190    3.7700
police_station                 object    298445    0.0000
data_sent_to_s

In [4]:
# work on a copy so raw data is always available
df = df_raw.copy()

## 2 · Drop Useless Columns

In [5]:
# ── 2a. Drop columns that are 100 % null ──────────────────────────────────────
all_null_cols = df.columns[df.isna().all()].tolist()

# also catch columns whose only value is the string 'NULL'
string_null_cols = [
    c for c in df.columns
    if df[c].astype(str).str.strip().str.upper().isin({'NULL', 'NAN', 'NONE', ''}).all()
]

drop_cols = list(set(all_null_cols + string_null_cols))
print('Dropping fully-null / string-NULL columns:', drop_cols)
df.drop(columns=drop_cols, inplace=True)

Dropping fully-null / string-NULL columns: ['closed_datetime', 'description', 'action_taken_timestamp']


In [6]:
# ── 2b. Drop columns with > 95 % missing (configurable threshold) ─────────────
MISSING_THRESHOLD = 0.95
high_miss = df.columns[(df.isna().mean() > MISSING_THRESHOLD)].tolist()
print(f'Columns with >{MISSING_THRESHOLD*100:.0f}% missing:', high_miss)
df.drop(columns=high_miss, inplace=True)

print(f'\nShape after column drops: {df.shape}')

Columns with >95% missing: []

Shape after column drops: (298450, 21)


## 3 · Deduplication

In [7]:
# ── 3a. Exact duplicates ──────────────────────────────────────────────────────
n_before = len(df)
df.drop_duplicates(inplace=True)
print(f'Exact duplicates removed: {n_before - len(df):,}')

Exact duplicates removed: 0


In [8]:
# ── 3b. Remove records flagged as duplicate in validation_status ───────────────
if 'validation_status' in df.columns:
    n_before = len(df)
    df['validation_status'] = df['validation_status'].astype(str).str.strip().str.lower()
    df = df[df['validation_status'] != 'duplicate'].copy()
    print(f'Validation-status duplicates removed: {n_before - len(df):,}')

Validation-status duplicates removed: 320


In [9]:
# ── 3c. Near-duplicate: same vehicle + same location + same device within 5 min ─
# Only runs if the key columns are present
dedup_key_cols = ['vehicle_number', 'latitude', 'longitude', 'device_id', 'created_datetime']
available_dedup = [c for c in dedup_key_cols if c in df.columns]

if 'created_datetime' in df.columns and len(available_dedup) >= 3:
    df['created_datetime'] = pd.to_datetime(df['created_datetime'], errors='coerce', utc=True)
    df_sorted = df.sort_values('created_datetime')

    # round timestamp to 5-minute bucket and deduplicate within it
    df_sorted['_time_bucket'] = df_sorted['created_datetime'].dt.floor('5min')
    group_cols = [c for c in ['vehicle_number', 'latitude', 'longitude', 'device_id', '_time_bucket'] if c in df_sorted.columns]

    n_before = len(df_sorted)
    df_sorted.drop_duplicates(subset=group_cols, keep='first', inplace=True)
    df_sorted.drop(columns=['_time_bucket'], inplace=True)
    df = df_sorted.copy()
    print(f'Near-duplicates (5-min window) removed: {n_before - len(df):,}')
else:
    print('Skipping near-duplicate check – required columns not all present.')

print(f'Shape after deduplication: {df.shape}')

Near-duplicates (5-min window) removed: 5,481
Shape after deduplication: (292649, 21)


## 4 · Geospatial Cleaning

In [10]:
# Bengaluru bounding box
LAT_MIN, LAT_MAX = 12.70, 13.20
LON_MIN, LON_MAX = 77.40, 77.80

if 'latitude' in df.columns and 'longitude' in df.columns:
    df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

    # flag (0, 0) GPS default
    zero_gps = (df['latitude'] == 0) & (df['longitude'] == 0)
    df.loc[zero_gps, ['latitude', 'longitude']] = np.nan
    print(f'(0,0) GPS defaults nulled out : {zero_gps.sum():,}')

    # flag out-of-Bengaluru coordinates
    out_of_bbox = (
        df['latitude'].notna() & (
            (df['latitude']  < LAT_MIN) | (df['latitude']  > LAT_MAX) |
            (df['longitude'] < LON_MIN) | (df['longitude'] > LON_MAX)
        )
    )
    df.loc[out_of_bbox, ['latitude', 'longitude']] = np.nan
    print(f'Out-of-bounding-box coords nulled: {out_of_bbox.sum():,}')

    print(f'Null lat/lon after clean: {df["latitude"].isna().sum():,}')

(0,0) GPS defaults nulled out : 0
Out-of-bounding-box coords nulled: 168
Null lat/lon after clean: 168


## 5 · Datetime Parsing & Validation

In [11]:
DATETIME_COLS = [
    'created_datetime', 'closed_datetime', 'modified_datetime',
    'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'validation_timestamp'
]
DATETIME_COLS = [c for c in DATETIME_COLS if c in df.columns]

NOW = pd.Timestamp.now(tz='UTC')

for col in DATETIME_COLS:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

# make sure all have timezone info (coerce to UTC)
for col in DATETIME_COLS:
    if df[col].dt.tz is None:
        df[col] = df[col].dt.tz_localize('UTC')

print('Datetime columns parsed:', DATETIME_COLS)

Datetime columns parsed: ['created_datetime', 'modified_datetime', 'data_sent_to_scita_timestamp', 'validation_timestamp']


In [12]:
if 'created_datetime' in df.columns:
    # future timestamps → null
    future_mask = df['created_datetime'] > NOW
    df.loc[future_mask, 'created_datetime'] = pd.NaT
    print(f'Future created_datetime nulled : {future_mask.sum():,}')

    # year 1900 artifacts → null
    past_mask = df['created_datetime'].dt.year < 2000
    df.loc[past_mask, 'created_datetime'] = pd.NaT
    print(f'Pre-2000 created_datetime nulled: {past_mask.sum():,}')

if 'closed_datetime' in df.columns and 'created_datetime' in df.columns:
    # closed must be >= created
    bad_close = df['closed_datetime'] < df['created_datetime']
    df.loc[bad_close, 'closed_datetime'] = pd.NaT
    print(f'Invalid closed < created nulled  : {bad_close.sum():,}')

Future created_datetime nulled : 0
Pre-2000 created_datetime nulled: 0


## 6 · JSON Array Parsing → Multi-Label Binarization

In [13]:
def safe_json_parse(val):
    """Parse a JSON array string; return empty list on failure."""
    if pd.isna(val) or str(val).strip() in ('', 'NULL', 'null', '[]'):
        return []
    try:
        parsed = json.loads(val)
        return [str(v).strip().upper() for v in parsed] if isinstance(parsed, list) else []
    except (json.JSONDecodeError, TypeError):
        return []

In [14]:
# ── violation_type → multi-hot columns ────────────────────────────────────────
if 'violation_type' in df.columns:
    df['violation_type_parsed'] = df['violation_type'].apply(safe_json_parse)

    mlb_viol = MultiLabelBinarizer()
    viol_encoded = pd.DataFrame(
        mlb_viol.fit_transform(df['violation_type_parsed']),
        columns=[f'viol_{v.replace(" ", "_").replace("/", "_")}' for v in mlb_viol.classes_],
        index=df.index
    )
    df = pd.concat([df, viol_encoded], axis=1)
    print(f'violation_type → {len(mlb_viol.classes_)} binary columns')
    print('  Labels:', list(mlb_viol.classes_))

violation_type → 27 binary columns
  Labels: ['2W/3W - USING MOBILE PHONE', 'AGAINST ONE WAY/NO ENTRY', 'CARRYING LENGHTY MATERIAL', 'DEFECTIVE NUMBER PLATE', 'DEMANDING EXCESS FARE', 'DOUBLE PARKING', 'FAIL TO USE SAFETY BELTS', 'H T V PROHIBITED', 'JUMPING TRAFFIC SIGNAL', 'NO PARKING', 'OBSTRUCTING DRIVER', 'OTHER - USING MOBILE PHONE', 'PARKING IN A MAIN ROAD', 'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC', 'PARKING NEAR ROAD CROSSING', 'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS', 'PARKING ON FOOTPATH', 'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE', 'PARKING OTHER THAN BUS STOP', 'REFUSE TO GO FOR HIRE', 'RIDER NOT WEARING HELMET', 'STOPING ON WHITE/STOP LINE', 'U TURN PROHIBITED', 'USING BLACK FILM/OTHER MATERIALS', 'VIOLATING LANE DISIPLINE', 'WITHOUT SIDE MIRROR', 'WRONG PARKING']


In [15]:
# ── offence_code → multi-hot columns ──────────────────────────────────────────
if 'offence_code' in df.columns:
    df['offence_code_parsed'] = df['offence_code'].apply(safe_json_parse)

    mlb_code = MultiLabelBinarizer()
    code_encoded = pd.DataFrame(
        mlb_code.fit_transform(df['offence_code_parsed']),
        columns=[f'code_{c}' for c in mlb_code.classes_],
        index=df.index
    )
    df = pd.concat([df, code_encoded], axis=1)
    print(f'offence_code → {len(mlb_code.classes_)} binary columns')

offence_code → 27 binary columns


## 7 · Text & Categorical Normalization

In [16]:
# ── vehicle_number: uppercase, strip hyphens/spaces ──────────────────────────
for col in ['vehicle_number', 'updated_vehicle_number']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.strip()
            .str.upper()
            .str.replace(r'[\s\-]', '', regex=True)
            .replace({'NAN': np.nan, 'NULL': np.nan, 'NONE': np.nan, '': np.nan})
        )

In [17]:
# ── vehicle_type / updated_vehicle_type: uppercase, strip ────────────────────
for col in ['vehicle_type', 'updated_vehicle_type']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str).str.strip().str.upper()
            .replace({'NAN': np.nan, 'NULL': np.nan, 'NONE': np.nan, '': np.nan})
        )

# Prefer updated_vehicle_type when available, fall back to vehicle_type
if 'updated_vehicle_type' in df.columns and 'vehicle_type' in df.columns:
    df['vehicle_type_clean'] = df['updated_vehicle_type'].fillna(df['vehicle_type'])
elif 'vehicle_type' in df.columns:
    df['vehicle_type_clean'] = df['vehicle_type']

print('vehicle_type_clean value counts:')
if 'vehicle_type_clean' in df.columns:
    print(df['vehicle_type_clean'].value_counts().to_string())

vehicle_type_clean value counts:
vehicle_type_clean
SCOOTER                93260
CAR                    86040
MOTOR CYCLE            40283
PASSENGER AUTO         37030
MAXI-CAB               11657
LGV                     8206
GOODS AUTO              2922
MOPED                   2069
PRIVATE BUS             1633
VAN                     1506
TEMPO                   1272
BUS (BMTC/KSRTC)        1238
HGV                     1161
LORRY/GOODS VEHICLE     1106
JEEP                     936
OTHERS                   877
TOURIST BUS              365
SCHOOL VEHICLE           361
TANKER                   253
FACTORY BUS              223
MINI LORRY               190
TRACTOR                   61


In [18]:
# ── police_station: strip, title-case ────────────────────────────────────────
if 'police_station' in df.columns:
    df['police_station'] = (
        df['police_station'].astype(str).str.strip()
        .replace({'NULL': np.nan, 'No Police Station': np.nan,
                  'none': np.nan, 'nan': np.nan, '': np.nan})
    )

In [19]:
# ── junction_name: keep NULL / No Junction as explicit category ───────────────
if 'junction_name' in df.columns:
    df['junction_name'] = (
        df['junction_name'].astype(str).str.strip()
        .replace({'NULL': 'No Junction', 'nan': 'No Junction', '': 'No Junction'})
    )
    # extract BTP code for easier grouping
    df['btp_code'] = df['junction_name'].str.extract(r'(BTP\d+)', expand=False)

In [20]:
# ── data_sent_to_scita → proper boolean ──────────────────────────────────────
if 'data_sent_to_scita' in df.columns:
    df['data_sent_to_scita'] = (
        df['data_sent_to_scita'].astype(str).str.strip().str.upper()
        .map({'TRUE': True, 'FALSE': False})
    )

# ── validation_status: lowercase, map created1 artifact ─────────────────────
if 'validation_status' in df.columns:
    df['validation_status'] = (
        df['validation_status'].astype(str).str.strip().str.lower()
        .replace({'null': np.nan, 'nan': np.nan, 'none': np.nan, '': np.nan,
                  'created1': 'created'})  # normalise the artifact value
    )

# ── center_code → integer ─────────────────────────────────────────────────────
if 'center_code' in df.columns:
    df['center_code'] = pd.to_numeric(df['center_code'], errors='coerce').astype('Int64')

## 8 · Derived / Engineered Features

In [21]:
if 'created_datetime' in df.columns:
    cd = df['created_datetime'].dt

    df['hour_of_day']  = cd.hour
    df['day_of_week']  = cd.dayofweek          # 0 = Monday
    df['day_name']     = cd.day_name()
    df['month']        = cd.month
    df['week_num']     = cd.isocalendar().week.astype('Int64')
    df['year']         = cd.year
    df['date']         = cd.date

    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype('Int64')
    df['is_peak_hour'] = df['hour_of_day'].apply(
        lambda h: 1 if (7 <= h <= 10 or 16 <= h <= 20) else 0
    ).astype('Int64')

    print('Temporal features created.')

Temporal features created.


In [22]:
# ── time-lag features ─────────────────────────────────────────────────────────
if 'closed_datetime' in df.columns and 'created_datetime' in df.columns:
    df['resolution_time_hrs'] = (
        (df['closed_datetime'] - df['created_datetime']).dt.total_seconds() / 3600
    ).round(2)
    # negative artefacts already nulled in step 5; clip to positive
    df.loc[df['resolution_time_hrs'] < 0, 'resolution_time_hrs'] = np.nan

if 'action_taken_timestamp' in df.columns and 'created_datetime' in df.columns:
    df['action_lag_mins'] = (
        (df['action_taken_timestamp'] - df['created_datetime']).dt.total_seconds() / 60
    ).round(2)
    df.loc[df['action_lag_mins'] < 0, 'action_lag_mins'] = np.nan

if 'data_sent_to_scita_timestamp' in df.columns and 'created_datetime' in df.columns:
    df['scita_latency_mins'] = (
        (df['data_sent_to_scita_timestamp'] - df['created_datetime']).dt.total_seconds() / 60
    ).round(2)
    df.loc[df['scita_latency_mins'] < 0, 'scita_latency_mins'] = np.nan

print('Time-lag features created.')

Time-lag features created.


In [23]:
# ── violation-count & severity features ───────────────────────────────────────
if 'violation_type_parsed' in df.columns:
    df['num_violations'] = df['violation_type_parsed'].apply(len)

    PARKING_VIOLATIONS = {
        'NO PARKING', 'WRONG PARKING', 'DOUBLE PARKING',
        'PARKING IN A MAIN ROAD', 'PARKING ON FOOTPATH',
        'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC',
        'PARKING NEAR ROAD CROSSING',
        'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS',
        'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE',
        'PARKING OTHER THAN BUS STOP'
    }
    HIGH_RISK_VIOLATIONS = {
        'DOUBLE PARKING', 'PARKING IN A MAIN ROAD',
        'OBSTRUCTING DRIVER', 'H T V PROHIBITED',
        'AGAINST ONE WAY/NO ENTRY'
    }

    df['has_parking_violation'] = df['violation_type_parsed'].apply(
        lambda v: 1 if bool(set(v) & PARKING_VIOLATIONS) else 0
    ).astype('Int64')

    df['is_high_risk_violation'] = df['violation_type_parsed'].apply(
        lambda v: 1 if bool(set(v) & HIGH_RISK_VIOLATIONS) else 0
    ).astype('Int64')

    print('Violation severity features created.')

Violation severity features created.


In [24]:
# ── vehicle class grouping ─────────────────────────────────────────────────────
if 'vehicle_type_clean' in df.columns:
    TWO_WHEELER  = {'MOTOR CYCLE', 'SCOOTER', 'MOPED'}
    THREE_WHEELER= {'PASSENGER AUTO', 'GOODS AUTO', 'TEMPO'}
    HEAVY        = {'HGV', 'LGV', 'LORRY/GOODS VEHICLE', 'TANKER', 'TRACTOR', 'MINI LORRY'}
    BUS          = {'BUS (BMTC/KSRTC)', 'PRIVATE BUS', 'TOURIST BUS', 'SCHOOL VEHICLE',
                    'FACTORY BUS', 'MAXI-CAB'}

    def classify_vehicle(vt):
        if pd.isna(vt): return 'UNKNOWN'
        if vt in TWO_WHEELER:   return '2W'
        if vt in THREE_WHEELER: return '3W'
        if vt in HEAVY:         return 'HV'
        if vt in BUS:           return 'BUS'
        if vt == 'CAR':         return 'CAR'
        if vt in {'JEEP', 'VAN'}: return 'LMV'
        return 'OTHER'

    df['vehicle_class'] = df['vehicle_type_clean'].apply(classify_vehicle)
    print('Vehicle class feature created.')

Vehicle class feature created.


## 9 · Null Audit & Final Summary

In [25]:
# ── drop intermediate helper columns used only during parsing ─────────────────
helper_cols = ['violation_type_parsed', 'offence_code_parsed']
df.drop(columns=[c for c in helper_cols if c in df.columns], inplace=True)

# ── reset index ───────────────────────────────────────────────────────────────
df.reset_index(drop=True, inplace=True)

print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Final shape: 292,649 rows × 91 columns


In [27]:
# ── null audit on core columns ─────────────────────────────────────────────────
core_cols = [
    'id', 'latitude', 'longitude', 'vehicle_number', 'vehicle_type_clean',
    'created_datetime', 'police_station', 'junction_name', 'validation_status',
    'num_violations', 'has_parking_violation'
]
core_cols = [c for c in core_cols if c in df.columns]

null_audit = pd.DataFrame({
    'null_count': df[core_cols].isna().sum(),
    'null_pct'  : (df[core_cols].isna().mean() * 100).round(2)
})
print('Null audit – core columns:')
print(null_audit.to_string())

Null audit – core columns:
                       null_count  null_pct
id                              0    0.0000
latitude                      168    0.0600
longitude                     168    0.0600
vehicle_number                  0    0.0000
vehicle_type_clean              0    0.0000
created_datetime                3    0.0000
police_station                341    0.1200
junction_name                   0    0.0000
validation_status          120425   41.1500
num_violations                  0    0.0000
has_parking_violation           0    0.0000


In [28]:
# ── final column list ──────────────────────────────────────────────────────────
print('All columns in cleaned dataframe:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:3}. {col}  [{df[col].dtype}]')

All columns in cleaned dataframe:
    1. id  [object]
    2. latitude  [float64]
    3. longitude  [float64]
    4. location  [object]
    5. vehicle_number  [object]
    6. vehicle_type  [object]
    7. violation_type  [object]
    8. offence_code  [object]
    9. created_datetime  [datetime64[ns, UTC]]
   10. modified_datetime  [datetime64[ns, UTC]]
   11. device_id  [object]
   12. created_by_id  [object]
   13. center_code  [Int64]
   14. police_station  [object]
   15. data_sent_to_scita  [bool]
   16. junction_name  [object]
   17. data_sent_to_scita_timestamp  [datetime64[ns, UTC]]
   18. updated_vehicle_number  [object]
   19. updated_vehicle_type  [object]
   20. validation_status  [object]
   21. validation_timestamp  [datetime64[ns, UTC]]
   22. viol_2W_3W_-_USING_MOBILE_PHONE  [int64]
   23. viol_AGAINST_ONE_WAY_NO_ENTRY  [int64]
   24. viol_CARRYING_LENGHTY_MATERIAL  [int64]
   25. viol_DEFECTIVE_NUMBER_PLATE  [int64]
   26. viol_DEMANDING_EXCESS_FARE  [int64]
   27. viol_

In [29]:
# ── quick sanity check ─────────────────────────────────────────────────────────
print('=== CLEANING SUMMARY ===')
print(f'  Raw rows        : {len(df_raw):,}')
print(f'  Clean rows      : {len(df):,}')
print(f'  Rows dropped    : {len(df_raw) - len(df):,}')
print(f'  Columns (final) : {df.shape[1]}')
df.describe(include='all').T

=== CLEANING SUMMARY ===
  Raw rows        : 298,450
  Clean rows      : 292,649
  Rows dropped    : 5,801
  Columns (final) : 91


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,292649,292649,FKID142734,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,292481.0000,NaN,NaN,NaN,12.9808,12.8027,12.9634,12.9773,12.9974,13.1999,0.0491
longitude,292481.0000,NaN,NaN,NaN,77.6003,77.4426,77.5712,77.5840,77.6202,77.7717,0.0505
location,289856,10934,"Unnamed Road, Begur Chikkanahalli, Bengaluru, Karnataka....",3998,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vehicle_number,292649,231884,FKN00GL4424,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
scita_latency_mins,41828.0000,NaN,NaN,NaN,28128.6800,21645.3700,25142.5100,25707.0750,26636.2925,219332.1800,14504.7718
num_violations,292649.0000,NaN,NaN,NaN,1.1671,1.0000,1.0000,1.0000,1.0000,12.0000,0.4828
has_parking_violation,292649.0000,<NA>,<NA>,<NA>,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
is_high_risk_violation,292649.0000,<NA>,<NA>,<NA>,0.0840,0.0000,0.0000,0.0000,0.0000,1.0000,0.2774


In [31]:
# ── save cleaned data ──────────────────────────────────────────────────────────
OUTPUT_PATH = 'parking_violations_clean.parquet'   # parquet keeps dtypes perfectly
OUTPUT_PATH = 'parking_violations_clean.csv'     # use this if you prefer CSV

df.to_parquet(OUTPUT_PATH, index=False)
# df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved → {OUTPUT_PATH}')

Saved → parking_violations_clean.csv
